In [60]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count
import numpy as np
import pandas as pd
import os

spark = SparkSession.builder \
    .appName("Pertemuan6-ParquetETL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [61]:
# Membuat dataset yang lebih besar (20.000 baris) khusus untuk latihan ini
np.random.seed(11)
n = 20000
data_besar = {
    "order_id": [f"O{i}" for i in range(n)],
    "customer_id": np.random.randint(1, 21, size=n),
    "kategori": np.random.choice(["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"], size=n),
    "kota": np.random.choice(["Magelang", "Semarang", "Solo", "Yogyakarta", "Purworejo"], size=n),
    "metode_pembayaran": np.random.choice(["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"], size=n),
    "pendapatan": np.random.randint(50000, 500000, size=n),
}
pd.DataFrame(data_besar).to_csv("data_transaksi_besar.csv", index=False)

csv_size = os.path.getsize("data_transaksi_besar.csv")
print(f"Ukuran CSV: {csv_size:,} bytes ({csv_size/1024:.1f} KB) untuk {n:,} baris")

Ukuran CSV: 892,909 bytes (872.0 KB) untuk 20,000 baris


In [62]:
# Membaca CSV, lalu menuliskannya ulang sebagai Parquet
df_besar = spark.read.csv(
    "data_transaksi_besar.csv",
    header=True,
    inferSchema=True
)

df_besar.write.mode("overwrite").parquet(
    "data_transaksi_besar_parquet"
)

# Menghitung total ukuran seluruh berkas .parquet yang dihasilkan
parquet_size = 0

for dp, dn, fn in os.walk("data_transaksi_besar_parquet"):
    for f in fn:
        if f.endswith(".parquet"):
            file_path = os.path.join(dp, f)
            parquet_size += os.path.getsize(file_path)

print(f"Ukuran CSV      : {csv_size:,} bytes ({csv_size/1024:.1f} KB)")
print(f"Ukuran Parquet  : {parquet_size:,} bytes ({parquet_size/1024:.1f} KB)")
print(f"Parquet {(1 - parquet_size/csv_size)*100:.1f}% lebih kecil dibanding CSV")

[Stage 14:>                                                         (0 + 1) / 1]

Ukuran CSV      : 892,909 bytes (872.0 KB)
Ukuran Parquet  : 202,197 bytes (197.5 KB)
Parquet 77.4% lebih kecil dibanding CSV


---
## Latihan Mandiri

Jalankan ulang cell pembuatan `SparkSession` dari awal modul, lalu baca kembali `data_transaksi_besar.csv` dan `data_pelanggan.json` sebelum mengerjakan latihan berikut.

In [63]:
# Persiapan ulang untuk latihan
spark = SparkSession.builder.appName("Latihan6").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df_transaksi_etl = spark.read.csv("data_transaksi_besar.csv", header=True, inferSchema=True)
df_pelanggan_etl = spark.read.json("data_pelanggan.json")
print("Siap untuk latihan.")

Siap untuk latihan.


**Soal 1.** Baca kembali `data_transaksi_besar_parquet` (hasil Sub-bab 6.1.1) menggunakan `spark.read.parquet()`, lalu tampilkan `printSchema()`-nya. Bandingkan dengan skema hasil `inferSchema=True` pada CSV — apakah tipe datanya sama?

In [64]:
#Membaca kembali data Parquet
df_parquet = spark.read.parquet("data_transaksi_besar_parquet")

#Menampilkan schema
df_parquet.printSchema()

#Menampilkan schema dari CSV sebagai pembanding
df_besar.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- pendapatan: integer (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- pendapatan: integer (nullable = true)



Jawaban: ya, tipe datanya sama (order_id: string, customer_id: int, kategori/kota/metode_pembayaran: string, pendapatan: int) karena Spark berhasil menebak tipe data CSV dengan benar saat file Parquet pertama kali ditulis. Bedanya, pada Parquet skema ini TERSIMPAN di dalam berkas, jadi saat dibaca ulang Spark tidak perlu menebak (inferSchema) lagi.

**Soal 2.** Gabungkan `df_transaksi_etl` dengan `df_pelanggan_etl`, lalu tampilkan jumlah transaksi untuk masing-masing nilai `membership` (`Silver`/`Gold`/`Platinum`).

In [65]:
#Menggabungkan data transaksi dengan data pelanggan
df_join = df_transaksi_etl.join(
    df_pelanggan_etl,
    on="customer_id",
    how="inner"
)

#Mengelompokkan berdasarkan membership, lalu menghitung jumlah transaksi tiap kelompok
df_membership = (
    df_join
    .groupBy("membership")
    .count()
    .orderBy("membership")
)

df_membership.show()

+----------+-----+
|membership|count|
+----------+-----+
|      Gold| 6990|
|  Platinum| 4016|
|    Silver| 8994|
+----------+-----+



**Soal 3.** Tulis ulang hasil gabungan Soal 2 (sebelum di-`groupBy`) ke disk lokal dalam format Parquet, dipartisi berdasarkan `membership`. Tampilkan struktur foldernya dengan `!ls`.

In [66]:
#Menyimpan hasil join ke Parquet dan dipartisi berdasarkan membership
df_join.write \
    .mode("overwrite") \
    .partitionBy("membership") \
    .parquet("transaksi_membership_parquet")

#Menampilkan struktur folder
!ls -R transaksi_membership_parquet

[Stage 24:>                                                         (0 + 1) / 1]

transaksi_membership_parquet:
'membership=Gold'  'membership=Platinum'  'membership=Silver'   _SUCCESS

'transaksi_membership_parquet/membership=Gold':
part-00000-06acceb7-f194-439a-8724-74de05169e63.c000.snappy.parquet

'transaksi_membership_parquet/membership=Platinum':
part-00000-06acceb7-f194-439a-8724-74de05169e63.c000.snappy.parquet

'transaksi_membership_parquet/membership=Silver':


part-00000-06acceb7-f194-439a-8724-74de05169e63.c000.snappy.parquet


**Soal 4 (Refleksi singkat).** Dalam 2-3 kalimat: mengapa partitioning berdasarkan kolom `kota` cocok untuk skenario kita, namun akan menjadi **pilihan buruk** jika diterapkan pada kolom seperti `order_id` yang nilainya selalu unik di setiap baris? Tulis jawaban pada markdown cell di bawah ini.

Partitioning berdasarkan kolom `kota` cocok karena nilai `kota` memiliki beberapa kategori yang berulang, sehingga data dapat dikelompokkan ke dalam sejumlah partisi yang masih terkontrol. Sebaliknya, `order_id` tidak cocok digunakan sebagai partition karena setiap nilainya unik, sehingga dapat menghasilkan terlalu banyak partisi dan membuat pengelolaan serta pembacaan data menjadi tidak efisien.


---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 7 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Tim data engineering platform e-commerce meminta anda membangun **pipeline ETL produksi pertama** yang menggabungkan tiga sumber data sekaligus: transaksi (CSV), data produk (JSON), dan data ulasan pelanggan (CSV terpisah) — merepresentasikan kondisi nyata di mana data tersebar di berbagai sistem dengan format berbeda-beda.

### Menyiapkan Dataset

Jalankan cell berikut untuk menghasilkan **tiga sumber data** sekaligus.

In [67]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


### Instruksi Pengerjaan

Buat notebook baru **`Tugas6_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession`, lalu bangun pipeline ETL lengkap mengikuti struktur **Extract → Transform → Load**:

---

**A. EXTRACT** *(bobot 15%)*

Baca ketiga sumber data (`tugas6_transaksi.csv`, `tugas6_produk.json`, `tugas6_ulasan.csv`) menjadi tiga Spark DataFrame terpisah. Tampilkan jumlah baris dan `printSchema()` masing-masing.

**B. TRANSFORM — Penggabungan** *(bobot 25%)*

Gabungkan ketiga DataFrame menjadi satu (`order_id` sebagai kunci ke ulasan, `product_id` sebagai kunci ke produk). Gunakan **`salah satu join yang tepat`** dari transaksi ke ulasan (karena tidak semua transaksi memiliki ulasan) dan **`salah satu join yang tepat`** dari transaksi ke produk (karena setiap transaksi pasti memiliki produk yang valid). Tambahkan kolom `total_pendapatan` (`unit_terjual x harga`).

**C. TRANSFORM — Penanganan Data Kosong & Pengayaan** *(bobot 20%)*

- Transaksi tanpa ulasan akan memiliki `rating` bernilai kosong (`null`) setelah `salah satu join yang tepat` — isi nilai kosong tersebut dengan angka **0** menggunakan `salah satu function`, sertakan alasan singkat mengapa 0 (bukan nilai lain) masuk akal untuk kasus "belum ada ulasan".
- Tambahkan kolom `ada_ulasan` bernilai `True`/`False` (tidak boleh diisi manual satu satu)`, **sebelum** langkah `na.fill()` di atas).

**D. LOAD** *(bobot 25%)*

Simpan hasil akhir ke HDFS dalam format **Parquet**, dipartisi berdasarkan `kategori`, ke path `/user/[username]/tugas6/hasil_etl`. Verifikasi dengan `hdfs dfs -ls -R`, lalu baca kembali dan tampilkan `count()`-nya sebagai bukti data tersimpan utuh.

**E. Insight Akhir** *(bobot 15%)*

Dari data hasil ETL, tampilkan (menggunakan DataFrame API **atau** Spark SQL, bebas memilih): kategori produk mana yang memiliki **persentase transaksi dengan ulasan** (`ada_ulasan = True`) **paling rendah**? Tulis 2-3 kalimat interpretasi bisnis pada markdown cell: mengapa hal ini mungkin penting diketahui oleh tim marketing?

---

### Ketentuan Pengumpulan

- Kumpulkan `Tugas6_[NPM]_[Nama Lengkap].ipynb` melalui Asprak, paling lambat **1 minggu dari hari ini, pukul 23.59 WIB**.
- Pastikan Hadoop aktif dan seluruh cell sudah dijalankan (**Run All**) sebelum dikumpulkan.

### Rubrik Penilaian

| Bagian | Kriteria | Bobot |
|---|---|---|
| A. Extract | Ketiga sumber berhasil dibaca dengan skema yang benar | 15% |
| B. Transform - Join | Kedua join (left & inner) diterapkan dengan tepat sesuai kebutuhan masing-masing | 25% |
| C. Transform - Data Quality | Missing value tertangani logis; kolom `ada_ulasan` benar | 20% |
| D. Load | Data tersimpan ke HDFS sebagai Parquet dengan partisi yang benar & terverifikasi | 25% |
| E. Insight Akhir | Analisis tepat & interpretasi bisnis relevan | 15% |


In [68]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, sum, round

# Membuat SparkSession
spark = SparkSession.builder \
    .appName("Tugas6_ETL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

## A. EXTRACT

In [69]:
#Membaca data transaksi
df_transaksi = spark.read.csv(
    "tugas6_transaksi.csv",
    header=True,
    inferSchema=True
)

#Membaca data produk
df_produk = spark.read.json("tugas6_produk.json")

#Membaca data ulasan
df_ulasan = spark.read.csv(
    "tugas6_ulasan.csv",
    header=True,
    inferSchema=True
)

#Menampilkan jumlah baris dan schema
print("=== DATA TRANSAKSI ===")
print(f"Jumlah baris: {df_transaksi.count()}")
df_transaksi.printSchema()

print("=== DATA PRODUK ===")
print(f"Jumlah baris: {df_produk.count()}")
df_produk.printSchema()

print("=== DATA ULASAN ===")
print(f"Jumlah baris: {df_ulasan.count()}")
df_ulasan.printSchema()

=== DATA TRANSAKSI ===
Jumlah baris: 5000
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

=== DATA PRODUK ===
Jumlah baris: 30
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

=== DATA ULASAN ===
Jumlah baris: 3500
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



## B. TRANSFORM — Penggabungan

- `df_transaksi` → `df_ulasan`: **left join** (kunci `order_id`), karena tidak semua transaksi memiliki ulasan — transaksi tanpa ulasan tetap harus dipertahankan.
- `df_transaksi` → `df_produk`: **inner join** (kunci `product_id`), karena setiap transaksi pasti punya `product_id` yang valid, jadi tidak ada risiko kehilangan baris.

In [70]:
#Join transaksi dengan ulasan
#Menggunakan left join karena tidak semua transaksi memiliki ulasan
df_join = df_transaksi.join(
    df_ulasan,
    on="order_id",
    how="left"
)

#Join hasil sebelumnya dengan data produk
#Menggunakan inner join karena setiap transaksi memiliki produk yang valid
df_join = df_join.join(
    df_produk,
    on="product_id",
    how="inner"
)

#Menambahkan kolom total_pendapatan
df_join = df_join.withColumn(
    "total_pendapatan",
    col("unit_terjual") * col("harga")
)

#Menampilkan hasil
df_join.show(10)

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|         1500000|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|          450000|
|        10|     TX5|           3|2026-10-01 00:00:00|     5| 75000|  Elektronik|  Produk-10|          225000|
|

## C. TRANSFORM — Penanganan Data Kosong & Pengayaan

Kolom `ada_ulasan` dibuat **sebelum** `na.fill()`, dari `rating.isNotNull()` — otomatis untuk seluruh baris, tanpa diisi manual. Nilai `rating` kosong lalu diisi **0** menggunakan `na.fill()`: masuk akal karena rating asli hanya berkisar 1-5 (tidak pernah 0), sehingga 0 dapat dibedakan dengan jelas sebagai penanda "belum ada ulasan", bukan rating rendah sungguhan.

In [71]:
#Menambahkan kolom ada_ulasan sebelum mengisi nilai null
df_join = df_join.withColumn(
    "ada_ulasan",
    col("rating").isNotNull()
)

#Mengisi rating yang kosong dengan 0
#Nilai 0 digunakan sebagai penanda bahwa transaksi belum memiliki ulasan, karena rating yang valid hanya berada pada rentang 1 sampai 5
df_join = df_join.na.fill({
    "rating": 0
})

#Menampilkan hasil
df_join.show(10)

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|ada_ulasan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|      true|
|         8|     TX1|           6|2026-10-25 00:00:00|     0|250000|     Fashion|   Produk-8|         1500000|     false|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|      true|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|      true|
|        19|     TX4|           6|2026-10-07 00:00:00|     0| 75000|Rumah Tangga|  Produk-19|          450000|     false|
|        10|     TX5|   

## D. LOAD

In [72]:
#Menentukan path penyimpanan di HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas6/hasil_etl

#Menyimpan hasil ETL dalam format Parquet
#Data dipartisi berdasarkan kategori
df_join.write \
    .mode("overwrite") \
    .partitionBy("kategori") \
    .parquet("hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl")

#Verifikasi struktur folder di HDFS
!hdfs dfs -ls -R /user/mahasiswa/tugas6/hasil_etl

#Membaca kembali data hasil ETL dari HDFS
df_hasil = spark.read.parquet("hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl")

#Menampilkan jumlah baris sebagai bukti data tersimpan
print(f"Jumlah baris hasil ETL: {df_hasil.count()}")

-rw-r--r--   3 dian-khaerunia-azzahra supergroup          0 2026-09-26 21:27 /user/mahasiswa/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - dian-khaerunia-azzahra supergroup          0 2026-09-26 21:27 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 dian-khaerunia-azzahra supergroup      16366 2026-09-26 21:27 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik/part-00000-26a5ce9b-a8db-4aa0-9586-da20d7a11422.c000.snappy.parquet
drwxr-xr-x   - dian-khaerunia-azzahra supergroup          0 2026-09-26 21:27 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 dian-khaerunia-azzahra supergroup      11133 2026-09-26 21:27 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion/part-00000-26a5ce9b-a8db-4aa0-9586-da20d7a11422.c000.snappy.parquet
drwxr-xr-x   - dian-khaerunia-azzahra supergroup          0 2026-09-26 21:27 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 dian-khaerunia-azzahra supergroup      10392 2026-09-26 21:27 /user/mahasiswa/tugas6/ha

## E. Insight Akhir

In [73]:
#Menghitung total transaksi dan transaksi yang memiliki ulasan
df_insight = df_hasil.groupBy("kategori").agg(
    count("*").alias("total_transaksi"),
    sum(
        when(col("ada_ulasan") == True, 1).otherwise(0)
    ).alias("transaksi_dengan_ulasan")
)

#Menghitung persentase transaksi yang memiliki ulasan
df_insight = df_insight.withColumn(
    "persentase_ulasan",
    round(
        col("transaksi_dengan_ulasan") /
        col("total_transaksi") * 100,
        2
    )
)

#Mengurutkan dari persentase ulasan paling rendah
df_insight = df_insight.orderBy("persentase_ulasan")

#Menampilkan hasil
df_insight.show()

#Menampilkan kategori dengan persentase ulasan paling rendah
kategori_terendah = df_insight.select(
    "kategori",
    "persentase_ulasan"
).first()

print(
    f"Kategori dengan persentase transaksi dengan ulasan paling rendah: "
    f"{kategori_terendah['kategori']} "
    f"({kategori_terendah['persentase_ulasan']}%)"
)

+------------+---------------+-----------------------+-----------------+
|    kategori|total_transaksi|transaksi_dengan_ulasan|persentase_ulasan|
+------------+---------------+-----------------------+-----------------+
|     Makanan|            536|                    369|            68.84|
|   Kesehatan|            961|                    662|            68.89|
|     Fashion|           1035|                    726|            70.14|
|Rumah Tangga|            815|                    575|            70.55|
|  Elektronik|           1653|                   1168|            70.66|
+------------+---------------+-----------------------+-----------------+

Kategori dengan persentase transaksi dengan ulasan paling rendah: Makanan (68.84%)


Kategori Makanan memiliki persentase transaksi dengan ulasan paling rendah (68.84%). Hal ini penting diketahui tim marketing karena produk makanan biasanya dikonsumsi dengan cepat dan pembeli cenderung lupa atau kurang terdorong untuk memberikan ulasan. Tim marketing dapat merancang kampanye seperti memberikan reward point atau voucher diskon khusus bagi pembeli kategori Makanan yang bersedia memberikan ulasan.   

In [74]:
#Menutup SparkSession setelah selesai
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
